In [1]:
# ============================================
# CELL 1: Install dependencies
# ============================================
import sys
import subprocess
import importlib

REQUIRED_PACKAGES = [
    "qdrant-client>=1.9.0",
    "sentence-transformers>=2.6.0",
    "transformers>=4.44.0",
    "accelerate>=0.33.0",
    "bitsandbytes>=0.43.1",
    "python-dotenv>=1.0.0",
    "pandas>=2.0.0",
]

def ensure_packages(packages):
    for pkg in packages:
        module_name = pkg.split("==")[0].split(">=")[0].replace("-", "_")
        try:
            importlib.import_module(module_name)
        except Exception:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure_packages(REQUIRED_PACKAGES)
print("Packages ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00
Packages ready.


In [2]:
# ============================================
# CELL 2: Imports and configuration
# ============================================
import os
import re
import json
import importlib
import torch
import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.http.models import Filter, FieldCondition, MatchValue, Range
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)

# ============================================
# Configuration helpers
# ============================================
def _read_kaggle_secret(key_name):
    try:
        kaggle_secrets = importlib.import_module("kaggle_secrets")
        user_secrets = kaggle_secrets.UserSecretsClient()
        return user_secrets.get_secret(key_name)
    except Exception:
        return None

def _get_config_value(secret_keys, env_keys=None, default=None):
    for key in secret_keys:
        val = _read_kaggle_secret(key)
        if val not in (None, ""):
            return val, f"secret:{key}"
    env_keys = env_keys or []
    for key in env_keys:
        val = os.getenv(key)
        if val not in (None, ""):
            return val, f"env:{key}"
    return default, "default"

# ============================================
# Load configuration
# ============================================
QDRANT_API_KEY, src_qdrant_key = _get_config_value(
    ["QDRAN_API_KEY", "QDRANT_API_KEY"],
    ["QDRAN_API_KEY", "QDRANT_API_KEY"],
)
QDRANT_URL, src_qdrant_url = _get_config_value(
    ["QDRAN_API_URL", "QDRANT_API_URL", "QDRANT_URL"],
    ["QDRAN_API_URL", "QDRANT_API_URL", "QDRANT_URL"],
)

if not QDRANT_URL:
    raise ValueError("Missing Qdrant URL. Set Kaggle Secret QDRANT_URL")

COLLECTION_PLACES = "places_danang"
COLLECTION_PLACE_REVIEWS = "place_reviews_danang"
COLLECTION_RESTAURANTS = "restaurants_danang"
COLLECTION_RESTAURANT_REVIEWS = "restaurant_reviews_danang"
COLLECTION_ACCOMMODATION_HOTELS = "accommodation_hotels_danang"
COLLECTION_ACCOMMODATION_ROOMS = "accommodation_rooms_danang"
COLLECTION_ACCOMMODATION_REVIEWS = "accommodation_reviews_danang"

EMBED_MODEL_NAME = "BAAI/bge-m3"
LOCAL_LLM_MODEL = "Qwen/Qwen2.5-3B-Instruct"
USE_4BIT = True

print("Qdrant URL:", QDRANT_URL)
print("Collections:", COLLECTION_PLACES, COLLECTION_RESTAURANTS, COLLECTION_ACCOMMODATION_HOTELS)
print("Embedding model:", EMBED_MODEL_NAME)
print("CUDA available:", torch.cuda.is_available())

Qdrant URL: https://511742a9-e9a3-4261-91dd-5dbb998f21a7.us-west-1-0.aws.cloud.qdrant.io:6333
Collections: places_danang restaurants_danang accommodation_hotels_danang
Embedding model: BAAI/bge-m3
CUDA available: True


In [3]:
# ============================================
# CELL 3: Define Query Intent and Data Models (FIXED)
# ============================================

class QueryIntent(Enum):
    """Query intent types for better retrieval"""
    HOTEL_SEARCH = "hotel_search"           # Tìm khách sạn
    RESTAURANT_SEARCH = "restaurant_search" # Tìm nhà hàng
    PLACE_SEARCH = "place_search"           # Tìm địa điểm du lịch
    REVIEW_SEARCH = "review_search"         # Tìm review/đánh giá
    ROOM_SEARCH = "room_search"             # Tìm thông tin phòng
    PRICE_SEARCH = "price_search"           # Tìm thông tin giá
    GENERAL = "general"                     # Truy vấn chung
    
    @classmethod
    def detect(cls, query: str) -> "QueryIntent":
        q = query.lower()
        
        # CRITICAL FIX: Check entity types FIRST before review/price
        # Because "khách sạn có đánh giá tốt" should be HOTEL, not REVIEW
        
        # Hotel specific (HIGHEST PRIORITY)
        if any(k in q for k in ["khách sạn", "hotel", "resort", "nghỉ dưỡng", "lưu trú"]):
            return cls.HOTEL_SEARCH
        
        # Restaurant specific
        if any(k in q for k in ["nhà hàng", "restaurant", "quán ăn", "ăn uống", "hải sản"]):
            return cls.RESTAURANT_SEARCH
        
        # Place specific
        if any(k in q for k in ["địa điểm", "thắng cảnh", "tham quan", "du lịch", "bãi biển"]):
            return cls.PLACE_SEARCH
        
        # Room related
        if any(k in q for k in ["phòng", "room", "giường", "sức chứa", "capacity"]):
            return cls.ROOM_SEARCH
        
        # Review related (only if NOT already caught by entity types)
        if any(k in q for k in ["review", "đánh giá", "nhận xét", "comment", "chất lượng"]):
            return cls.REVIEW_SEARCH
        
        # Price related (last priority)
        if any(k in q for k in ["giá", "price", "bao nhiêu", "chi phí", "budget"]):
            return cls.PRICE_SEARCH
        
        return cls.GENERAL

# ====================== THÊM MỚI: FILTER PARSER ======================
def parse_filters_from_query(query: str) -> Dict:
    """Tự động trích xuất filter từ câu query tự nhiên"""
    q = query.lower().strip()
    filters = {}
    
    # === Parse District ===
    district_map = {
        "sơn trà": "son tra", "son tra": "son tra",
        "hải châu": "hai chau", "hai chau": "hai chau",
        "ngũ hành sơn": "ngu hanh son", "cẩm lệ": "cam le",
        "liên chiểu": "lien chieu", "thanh khê": "thanh khe"
    }
    for vn, code in district_map.items():
        if vn in q:
            filters["district"] = code
            break
    
    # === Parse Price ===
    import re
    # Tìm "dưới X triệu", "dưới X", "< X", "max X", "giá rẻ",...
    price_patterns = [
        r"dưới\s*(\d+)\s*(triệu|tr)",      # dưới 1 triệu
        r"duoi\s*(\d+)\s*(tr|triệu)",
        r"giá\s*<\s*(\d+)", 
        r"max\s*(\d+)", 
        r"nhỏ hơn\s*(\d+)",
        r"rẻ hơn\s*(\d+)"
    ]
    
    for pattern in price_patterns:
        match = re.search(pattern, q)
        if match:
            value = int(match.group(1))
            if "triệu" in q or "tr" in q:
                value *= 1_000_000
            filters["max_price"] = value
            break
    
    # === Parse Min Rating ===
    rating_match = re.search(r"(trên|>=|>\s*)(\d+\.?\d*)\s*(điểm| sao)", q)
    if rating_match:
        filters["min_rating"] = float(rating_match.group(2))
    
    return filters
@dataclass
class SearchResult:
    """Structured search result"""
    point_id: str
    collection: str
    score: float
    entity_name: str = ""
    place_name: str = ""
    district: str = ""
    rating: Optional[float] = None
    min_price: Optional[float] = None
    max_price: Optional[float] = None
    address: str = ""
    content: str = ""
    parent_entity_name: Optional[str] = None
    parent_entity_id: Optional[str] = None
    parent_rating: Optional[float] = None  # NEW: for review parent rating
    parent_address: Optional[str] = None   # NEW: for review parent address
    room_name: Optional[str] = None
    capacity: Optional[int] = None
    bed_type: Optional[str] = None
    area_m2: Optional[float] = None
    room_view: Optional[str] = None
    cuisine: Optional[str] = None
    restaurant_type: Optional[str] = None
    check_in_time: Optional[str] = None
    check_out_time: Optional[str] = None
    time_open: Optional[str] = None
    time_close: Optional[str] = None
    tags: Optional[List[str]] = None
    review_count: Optional[int] = None
    star_rating: Optional[float] = None
    price_level: Optional[str] = None
    price_currency: Optional[str] = None
    
    def to_dict(self) -> Dict:
        return {k: v for k, v in self.__dict__.items() if v is not None}
    
    def get_display_name(self) -> str:
        """Get best available name for display - FIXED for reviews"""
        # For reviews, ALWAYS use parent entity name first
        if self.collection in [
            COLLECTION_ACCOMMODATION_REVIEWS, 
            COLLECTION_RESTAURANT_REVIEWS, 
            COLLECTION_PLACE_REVIEWS
        ]:
            if self.parent_entity_name and self.parent_entity_name not in ["None", "", "null"]:
                return self.parent_entity_name
            if self.entity_name and self.entity_name not in ["None", "", "null"]:
                return self.entity_name
            if self.place_name and self.place_name not in ["None", "", "null"]:
                return self.place_name
            return "Đang cập nhật"
        
        # For entities
        if self.entity_name and self.entity_name not in ["None", "", "null"]:
            return self.entity_name
        if self.place_name and self.place_name not in ["None", "", "null"]:
            return self.place_name
        return "Unknown"
    
    def get_price_display(self) -> str:
        """Format price for display"""
        if self.min_price is None:
            return "Không có thông tin giá"
        if self.max_price and self.max_price > self.min_price:
            return f"{self.min_price:,.0f} - {self.max_price:,.0f} VND"
        return f"{self.min_price:,.0f} VND"
    
    def get_rating_display(self) -> str:
        """Format rating for display - FIXED for reviews"""
        # For reviews, use parent rating if available
        rating_value = self.parent_rating if self.parent_rating else self.rating
        
        if rating_value is None:
            return "Chưa có đánh giá"
        
        if self.review_count and self.review_count > 0:
            return f"{rating_value:.1f}/10 ({self.review_count:,} đánh giá)"
        return f"{rating_value:.1f}/10"
    
    def get_address_display(self) -> str:
        """Get address for display - FIXED for reviews"""
        # For reviews, use parent address if available
        if self.collection in [
            COLLECTION_ACCOMMODATION_REVIEWS, 
            COLLECTION_RESTAURANT_REVIEWS, 
            COLLECTION_PLACE_REVIEWS
        ]:
            if self.parent_address and self.parent_address not in ["None", "", "null"]:
                return self.parent_address
        return self.address if self.address not in ["None", "", "null"] else "Chưa có địa chỉ"


class CollectionRegistry:
    """Registry for all collections with metadata"""
    
    COLLECTIONS = {
        COLLECTION_PLACES: {
            "type": "place",
            "weight": 1.2,
            "display_name": "Địa điểm du lịch",
            "priority_for_intent": {
                QueryIntent.PLACE_SEARCH: 1.5,
                QueryIntent.GENERAL: 1.2,
            }
        },
        COLLECTION_RESTAURANTS: {
            "type": "restaurant",
            "weight": 1.2,
            "display_name": "Nhà hàng",
            "priority_for_intent": {
                QueryIntent.RESTAURANT_SEARCH: 1.5,
                QueryIntent.GENERAL: 1.2,
            }
        },
        COLLECTION_ACCOMMODATION_HOTELS: {
            "type": "hotel",
            "weight": 1.3,
            "display_name": "Khách sạn",
            "priority_for_intent": {
                QueryIntent.HOTEL_SEARCH: 1.6,
                QueryIntent.ROOM_SEARCH: 1.4,
                QueryIntent.PRICE_SEARCH: 1.3,
                QueryIntent.GENERAL: 1.3,
            }
        },
        COLLECTION_PLACE_REVIEWS: {
            "type": "review",
            "weight": 1.0,
            "display_name": "Đánh giá địa điểm",
            "priority_for_intent": {
                QueryIntent.REVIEW_SEARCH: 1.6,
            }
        },
        COLLECTION_RESTAURANT_REVIEWS: {
            "type": "review",
            "weight": 1.0,
            "display_name": "Đánh giá nhà hàng",
            "priority_for_intent": {
                QueryIntent.REVIEW_SEARCH: 1.6,
            }
        },
        COLLECTION_ACCOMMODATION_REVIEWS: {
            "type": "review",
            "weight": 1.0,
            "display_name": "Đánh giá khách sạn",
            "priority_for_intent": {
                QueryIntent.REVIEW_SEARCH: 1.6,
            }
        },
        COLLECTION_ACCOMMODATION_ROOMS: {
            "type": "room",
            "weight": 0.95,
            "display_name": "Thông tin phòng",
            "priority_for_intent": {
                QueryIntent.ROOM_SEARCH: 1.8,
            }
        },
    }
    
    @classmethod
    def get_collections_by_intent(cls, intent: QueryIntent) -> List[str]:
        """Get relevant collections for an intent - FIXED to avoid mixing"""
        if intent == QueryIntent.HOTEL_SEARCH:
            # ONLY hotels, no restaurants
            return [COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_ACCOMMODATION_REVIEWS, COLLECTION_ACCOMMODATION_ROOMS]
        elif intent == QueryIntent.RESTAURANT_SEARCH:
            # ONLY restaurants
            return [COLLECTION_RESTAURANTS, COLLECTION_RESTAURANT_REVIEWS]
        elif intent == QueryIntent.PLACE_SEARCH:
            # ONLY places
            return [COLLECTION_PLACES, COLLECTION_PLACE_REVIEWS]
        elif intent == QueryIntent.REVIEW_SEARCH:
            return [COLLECTION_PLACE_REVIEWS, COLLECTION_RESTAURANT_REVIEWS, COLLECTION_ACCOMMODATION_REVIEWS]
        elif intent == QueryIntent.ROOM_SEARCH:
            return [COLLECTION_ACCOMMODATION_ROOMS, COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_ACCOMMODATION_REVIEWS]
        elif intent == QueryIntent.PRICE_SEARCH:
            return [COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_RESTAURANTS]
        else:
            return [COLLECTION_ACCOMMODATION_HOTELS, COLLECTION_RESTAURANTS, COLLECTION_PLACES]
    
    @classmethod
    def get_weight(cls, collection: str, intent: QueryIntent) -> float:
        """Get weight for a collection based on intent"""
        config = cls.COLLECTIONS.get(collection, {"weight": 1.0})
        base_weight = config["weight"]
        priority = config.get("priority_for_intent", {}).get(intent, 1.0)
        return base_weight * priority

In [4]:
# ============================================
# CELL 4: Qdrant client and retrieval functions (FIXED)
# ============================================

# Connect to Qdrant
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=60)

# Load embedding model
encoder = SentenceTransformer(EMBED_MODEL_NAME, device="cuda" if torch.cuda.is_available() else "cpu")

# ===== THÊM VÀO: Cache và fetch parent function =====
_parent_cache = {}

def fetch_parent_entity(parent_id: str, collection_name: str) -> Optional[Dict]:
    """Fetch parent entity by ID with caching"""
    if not parent_id or parent_id in ["None", "", "null"]:
        return None
    
    if parent_id in _parent_cache:
        return _parent_cache[parent_id]
    
    # Determine which collection to query based on review type
    if "accommodation" in collection_name or "hotel" in collection_name:
        parent_collection = COLLECTION_ACCOMMODATION_HOTELS
    elif "restaurant" in collection_name:
        parent_collection = COLLECTION_RESTAURANTS
    elif "place" in collection_name:
        parent_collection = COLLECTION_PLACES
    else:
        return None
    
    try:
        points = client.retrieve(
            collection_name=parent_collection,
            ids=[parent_id],
            with_payload=True,
        )
        
        if points:
            payload = points[0].payload or {}
            result = {
                "name": payload.get("entity_name") or payload.get("place_name"),
                "rating": payload.get("rating"),
                "address": payload.get("address"),
                "district": payload.get("district"),
            }
            _parent_cache[parent_id] = result
            return result
    except Exception as e:
        print(f"Warning: Cannot fetch parent {parent_id}: {e}")
    
    return None


def extract_payload(point) -> Dict:
    """Extract payload from Qdrant point"""
    if hasattr(point, "payload"):
        return point.payload or {}
    if isinstance(point, dict):
        return point.get("payload", {})
    return {}

def extract_score(point) -> float:
    """Extract score from Qdrant point"""
    if hasattr(point, "score"):
        return point.score
    if isinstance(point, dict):
        return point.get("score", 0.0)
    return 0.0

def retrieve_from_collection(
    collection_name: str,
    query_vector: List[float],
    top_k: int = 5,
    filters: Optional[Dict] = None,
    score_threshold: float = 0.3
) -> List[SearchResult]:
    """Retrieve from a single collection with filters"""
    try:
       
        # Build filter conditions
        conditions = []
        if filters:
            if filters.get("district"):
                conditions.append(
                    FieldCondition(
                        key="district",
                        match=MatchValue(value=str(filters["district"]).lower().strip())
                    )
                )
            if filters.get("min_rating"):
                conditions.append(
                    FieldCondition(
                        key="rating",
                        range=Range(gte=float(filters["min_rating"]))
                    )
                )
            if filters.get("max_price"):
                conditions.append(
                    FieldCondition(
                        key="min_price_vnd",
                        range=Range(lte=float(filters["max_price"]))
                    )
                )
                print(f"  [DEBUG] {collection_name}: đã thêm condition max_price <= {filters['max_price']}")
            if filters.get("min_price"):
                conditions.append(
                    FieldCondition(
                        key="min_price_vnd",
                        range=Range(gte=float(filters["min_price"]))
                    )
                )
        
        # Chỉ tạo filter nếu có conditions
        q_filter = Filter(must=conditions) if conditions else None
        
      
        # Query Qdrant
        result = client.query_points(
            collection_name=collection_name,
            query=query_vector,
            query_filter=q_filter,
            limit=top_k,
            with_payload=True,
            with_vectors=False,
        )
        
        points = result.points if hasattr(result, "points") else result
        
     
        
        # Convert to SearchResult objects
        results = []
        price_filtered_count = 0
        
        for point in points:
            score = extract_score(point)
            if score < score_threshold:
                continue
                
            payload = extract_payload(point)
            
            # Lọc thủ công
            skip = False
            if filters:
                if filters.get("max_price"):
                    min_price = payload.get("min_price_vnd")
                    if min_price and min_price > filters["max_price"]:
                        skip = True
                        price_filtered_count += 1
                        print(f"  [DEBUG] {collection_name}: LOẠI {payload.get('entity_name')} - giá {min_price} > {filters['max_price']}")
                if filters.get("min_price"):
                    min_price = payload.get("min_price_vnd")
                    if min_price and min_price < filters["min_price"]:
                        skip = True
            
            if skip:
                continue
            
            result_obj = SearchResult(
                point_id=str(point.id) if hasattr(point, "id") else str(payload.get("id")),
                collection=collection_name,
                score=score,
                entity_name=payload.get("entity_name", ""),
                place_name=payload.get("place_name", ""),
                district=payload.get("district", ""),
                rating=float(payload.get("rating")) if payload.get("rating") else None,
                min_price=float(payload.get("min_price_vnd")) if payload.get("min_price_vnd") else None,
                max_price=float(payload.get("max_price_vnd")) if payload.get("max_price_vnd") else None,
                address=payload.get("address", ""),
                content=payload.get("content", ""),
                parent_entity_name=payload.get("parent_entity_name"),
                parent_entity_id=payload.get("parent_entity_id"),
                parent_rating=None,
                parent_address=None,
                room_name=payload.get("room_name"),
                capacity=int(payload.get("capacity")) if payload.get("capacity") else None,
                bed_type=payload.get("bed_type"),
                area_m2=float(payload.get("area_m2")) if payload.get("area_m2") else None,
                room_view=payload.get("room_view"),
                cuisine=payload.get("cuisine"),
                restaurant_type=payload.get("restaurant_type"),
                check_in_time=payload.get("check_in_time"),
                check_out_time=payload.get("check_out_time"),
                time_open=payload.get("time_open"),
                time_close=payload.get("time_close"),
                tags=payload.get("tags"),
                review_count=int(payload.get("review_count")) if payload.get("review_count") else None,
                star_rating=float(payload.get("star_rating")) if payload.get("star_rating") else None,
                price_level=payload.get("price_level"),
                price_currency=payload.get("price_currency"),
            )
            
            # For review collections, fetch parent entity info
            if collection_name in [COLLECTION_ACCOMMODATION_REVIEWS, COLLECTION_RESTAURANT_REVIEWS, COLLECTION_PLACE_REVIEWS]:
                parent_id = result_obj.parent_entity_id
                if parent_id and parent_id not in ["None", "", "null"]:
                    parent = fetch_parent_entity(parent_id, collection_name)
                    if parent:
                        if parent.get("name") and parent.get("name") not in ["None", "", "null"]:
                            result_obj.parent_entity_name = parent.get("name")
                        if parent.get("rating"):
                            result_obj.parent_rating = parent.get("rating")
                        if parent.get("address"):
                            result_obj.parent_address = parent.get("address")
                        if parent.get("district") and not result_obj.district:
                            result_obj.district = parent.get("district")
            
            results.append(result_obj)
        
        
        return results
        
    except Exception as e:
        print(f"Warning: Error retrieving from {collection_name}: {e}")
        return []


def retrieve_by_intent(
    query: str,
    intent: Optional[QueryIntent] = None,
    top_k_per_collection: int = 5,
    max_total_results: int = 20,
    filters: Optional[Dict] = None
) -> List[SearchResult]:
    """Main retrieval function with intent-based routing"""
    
    
    # Detect intent if not provided
    if intent is None:
        intent = QueryIntent.detect(query)
    
    # Get relevant collections
    collections = CollectionRegistry.get_collections_by_intent(intent)
    
    # Encode query
    query_vector = encoder.encode([query], normalize_embeddings=True)[0].tolist()
    
    # Retrieve from all relevant collections in parallel
    all_results = []
    
    def retrieve_single(collection):
        return retrieve_from_collection(
            collection_name=collection,
            query_vector=query_vector,
            top_k=top_k_per_collection,
            filters=filters  # ← QUAN TRỌNG: phải truyền filters
        )
    
    with ThreadPoolExecutor(max_workers=min(4, len(collections))) as executor:
        future_to_collection = {
            executor.submit(retrieve_single, collection): collection
            for collection in collections
        }
        for future in as_completed(future_to_collection):
            try:
                results = future.result()
                all_results.extend(results)
            except Exception as e:
                print(f"Error retrieving: {e}")
    
    # Apply reranking
    ranked_results = rerank_results(all_results, query, intent)
    
    # Lọc lại theo giá một lần nữa
    if filters and filters.get("max_price"):
        before_filter = len(ranked_results)
        ranked_results = [r for r in ranked_results if r.min_price and r.min_price <= filters["max_price"]]
    
    # Return top results
    return ranked_results[:max_total_results]

def rerank_results(
    results: List[SearchResult],
    query: str,
    intent: QueryIntent
) -> List[SearchResult]:
    """Rerank results using multiple signals"""
    
    query_lower = query.lower()
    
    for result in results:
        # Base score from Qdrant
        final_score = result.score
        
        # Apply collection weight
        collection_weight = CollectionRegistry.get_weight(result.collection, intent)
        final_score *= collection_weight
        
        # Boost for entities with ratings (use parent rating for reviews)
        rating_value = result.parent_rating if result.parent_rating else result.rating
        if rating_value is not None:
            rating_boost = min(0.5, rating_value / 20)  # Max 0.5 boost
            final_score += rating_boost
        
        # Boost for matching district
        if result.district and result.district.lower() in query_lower:
            final_score += 0.15
        
        # Intent-specific boosts
        if intent == QueryIntent.PRICE_SEARCH and result.min_price is not None:
            final_score += 0.2
        
        if intent == QueryIntent.ROOM_SEARCH and result.capacity is not None:
            final_score += 0.25
        
        if intent == QueryIntent.REVIEW_SEARCH and result.content:
            final_score += min(0.2, len(result.content) / 1000)
        
        # Boost for complete information
        info_score = 0
        if rating_value is not None:
            info_score += 0.05
        if result.min_price is not None:
            info_score += 0.05
        if result.address:
            info_score += 0.03
        if result.district:
            info_score += 0.03
        final_score += min(0.2, info_score)
        
        result.score = final_score
    
    # Sort by final score
    results.sort(key=lambda x: x.score, reverse=True)
    
    return results

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [5]:
# ============================================
# CELL 5: LLM Setup
# ============================================

# Load LLM
if torch.cuda.is_available() and USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
else:
    quant_config = None

tokenizer = AutoTokenizer.from_pretrained(LOCAL_LLM_MODEL, trust_remote_code=True)

model_kwargs = {
    "trust_remote_code": True,
    "device_map": "auto",
}

if torch.cuda.is_available():
    model_kwargs["dtype"] = torch.float16

if quant_config is not None:
    model_kwargs["quantization_config"] = quant_config

model = AutoModelForCausalLM.from_pretrained(LOCAL_LLM_MODEL, **model_kwargs)
model.eval()

if tokenizer.pad_token is None and tokenizer.eos_token is not None:
    tokenizer.pad_token = tokenizer.eos_token

print("Local LLM ready:", LOCAL_LLM_MODEL)


def generate_answer(
    query: str,
    results: List[SearchResult],
    max_new_tokens: int = 512,
    temperature: float = 0.2
) -> str:
    """Generate answer using LLM with retrieved context"""
    
    # Format context for LLM
    context_parts = []
    for i, r in enumerate(results[:8], 1):
        context = f"""
[{i}] {r.get_display_name()}
   - Loại: {r.collection}
   - Quận/Huyện: {r.district or 'Chưa có'}
   - Đánh giá: {r.get_rating_display()}
   - Giá: {r.get_price_display()}
   - Địa chỉ: {r.address or 'Chưa có'}"""
        
        if r.content:
            context += f"\n   - Nội dung: {r.content[:300]}..."
        
        if r.room_name:
            context += f"\n   - Phòng: {r.room_name}"
        if r.capacity:
            context += f" (Sức chứa: {r.capacity} người)"
        
        context_parts.append(context)
    
    context_text = "\n".join(context_parts)
    
    # Create prompt
    prompt = f"""Bạn là trợ lý du lịch thông minh tại Đà Nẵng. Hãy trả lời câu hỏi của người dùng dựa trên thông tin có sẵn.

### Câu hỏi:
{query}

### Thông tin tham khảo:
{context_text if context_text else "Không có thông tin phù hợp."}

### Hướng dẫn:
1. Trả lời trực tiếp, chính xác, trung thực dựa trên thông tin có sẵn
2. Nếu có nhiều lựa chọn, đề xuất TOP 3 phù hợp nhất
3. Luôn kèm theo đánh giá (rating) và giá (nếu có)
4. Nếu thông tin không đủ, hãy nói rõ phần còn thiếu
5. Sử dụng tiếng Việt, thân thiện và hữu ích

### Trả lời:"""

    # Format messages
    messages = [
        {"role": "system", "content": "Bạn là trợ lý du lịch Đà Nẵng chuyên nghiệp, thân thiện."},
        {"role": "user", "content": prompt},
    ]

    # Apply chat template
    if hasattr(tokenizer, "apply_chat_template"):
        try:
            model_inputs = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt",
                return_dict=True,
            )
        except TypeError:
            input_ids = tokenizer.apply_chat_template(
                messages,
                add_generation_prompt=True,
                return_tensors="pt",
            )
            model_inputs = {"input_ids": input_ids}
    else:
        prompt_text = f"System: {messages[0]['content']}\nUser: {messages[1]['content']}\nAssistant:"
        model_inputs = tokenizer(prompt_text, return_tensors="pt")

    # Move to device
    if hasattr(model_inputs, "to"):
        model_inputs = model_inputs.to(model.device)
    else:
        model_inputs = {
            k: (v.to(model.device) if hasattr(v, "to") else v)
            for k, v in model_inputs.items()
        }

    prompt_len = model_inputs["input_ids"].shape[-1]

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    gen_tokens = output_ids[0][prompt_len:]
    answer = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    
    return answer

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Local LLM ready: Qwen/Qwen2.5-3B-Instruct


In [6]:
# ============================================
# CELL 6: Main RAG Pipeline (FIXED)
# ============================================

class RAGPipeline:
    """Complete RAG pipeline orchestrator"""
    
    def __init__(self):
        self.encoder = encoder
        self.client = client
        self.default_top_k = 5
        self.default_max_results = 15
    
    def search(
        self,
        query: str,
        intent: Optional[QueryIntent] = None,
        top_k: int = 5,
        filters: Optional[Dict] = None,
        score_threshold: float = 0.3
    ) -> Tuple[List[SearchResult], QueryIntent]:
        
        # === THÊM PHẦN NÀY ===
        if filters is None:
            filters = parse_filters_from_query(query)   # Tự động parse
        
        # Detect intent
        intent = intent or QueryIntent.detect(query)
        
        # Retrieve
        results = retrieve_by_intent(
            query=query,
            intent=intent,
            top_k_per_collection=top_k,
            max_total_results=self.default_max_results,
            filters=filters          # ← Truyền filter vào
        )
        
        results = [r for r in results if r.score >= score_threshold]
        return results, intent
    
    def answer(
        self,
        query: str,
        top_k: int = 5,
        filters: Optional[Dict] = None,
        max_tokens: int = 512,
        temperature: float = 0.2
    ) -> Dict[str, Any]:
        """Get answer with context"""
        
        # Search - filters đã được truyền vào
        results, intent = self.search(query, top_k=top_k, filters=filters)
        
        # Generate answer
        answer = generate_answer(query, results, max_new_tokens=max_tokens, temperature=temperature)
        
        # Format results for display
        formatted_results = []
        for r in results[:10]:
            # CHỈ HIỂN THỊ KẾT QUẢ ĐÚNG VỚI FILTER
            if filters and filters.get("max_price"):
                if r.min_price and r.min_price > filters["max_price"]:
                    continue  # Bỏ qua kết quả không thỏa mãn
            
            formatted_results.append({
                "name": r.get_display_name(),
                "type": r.collection,
                "district": r.district,
                "rating": r.get_rating_display(),
                "price": r.get_price_display(),
                "address": r.address,
                "score": round(r.score, 3),
                "details": {
                    "room_name": r.room_name,
                    "capacity": r.capacity,
                    "cuisine": r.cuisine,
                    "check_in": r.check_in_time,
                    "check_out": r.check_out_time,
                }
            })
        
        return {
            "query": query,
            "intent": intent.value,
            "answer": answer,
            "results": formatted_results,
            "total_results": len(results)
        }

In [7]:
# ============================================
# CELL 7: Demo - Comprehensive Testing (ĐÃ SỬA)
# ============================================

print("=" * 80)
print("RAG SYSTEM DEMO - ĐÀ NẴNG TRAVEL ASSISTANT")
print("=" * 80)

# Đảm bảo pipeline luôn được khởi tạo
if 'pipeline' not in globals() or pipeline is None:
    print("🔄 Khởi tạo RAG Pipeline...")
    pipeline = RAGPipeline()
    print("✅ Pipeline initialized successfully!")

# Test queries
test_queries = [

    # ==================================================
    # KHÁCH SẠN
    # ==================================================
    
    ("🏨 Khách sạn", 
     "Gợi ý khách sạn ở quận Sơn Trà có đánh giá tốt trên 8.0"),

    ("🏨 Khách sạn giá rẻ", 
     "Khách sạn nào ở Đà Nẵng có giá dưới 1 triệu đồng?"),

    ("🏨 Khách sạn cao cấp", 
     "Khách sạn 5 sao nào ở Đà Nẵng được đánh giá cao?"),

    ("🏨 View biển", 
     "Khách sạn nào gần biển Mỹ Khê và có view đẹp?"),

    ("🏨 Gia đình", 
     "Khách sạn phù hợp cho gia đình 4 người ở Đà Nẵng"),

    ("🏨 Cặp đôi", 
     "Resort nào phù hợp cho cặp đôi nghỉ dưỡng?"),

    ("🏨 Hồ bơi", 
     "Khách sạn nào có hồ bơi vô cực đẹp?"),

    ("🏨 Gần trung tâm", 
     "Khách sạn gần cầu Rồng và trung tâm thành phố"),

    ("🏨 Review", 
     "Review về khách sạn Le Sands Oceanfront Danang Hotel "),

    ("🏨 Giá phòng", 
     "Giá phòng khách sạn ở Đà Nẵng khoảng bao nhiêu?"),

    ("🏨 Phòng lớn", 
     "Khách sạn nào có phòng gia đình sức chứa 4-5 người?"),

    ("🏨 Buffet sáng", 
     "Khách sạn nào có buffet sáng ngon?"),

    ("🏨 Check-in đẹp", 
     "Khách sạn nào có không gian check-in sống ảo đẹp?"),

    ("🏨 Nghỉ dưỡng", 
     "Gợi ý resort yên tĩnh để nghỉ dưỡng cuối tuần"),

    # ==================================================
    # NHÀ HÀNG
    # ==================================================

    ("🍽️ Nhà hàng", 
     "Nhà hàng nào ở Hải Châu có không gian đẹp và đồ ăn ngon?"),

    ("🍽️ Hải sản", 
     "Gợi ý nhà hàng hải sản ngon ở Đà Nẵng"),

    ("🍽️ Mì Quảng", 
     "Quán mì Quảng nổi tiếng ở Đà Nẵng"),

    ("🍽️ Bún chả cá", 
     "Ăn bún chả cá ngon ở đâu tại Đà Nẵng?"),

    ("🍽️ Đặc sản", 
     "Nhà hàng nào bán đặc sản Đà Nẵng ngon?"),

    ("🍽️ Buffet", 
     "Buffet nướng hải sản nào ngon và giá hợp lý?"),

    ("🍽️ Sang trọng", 
     "Nhà hàng sang trọng thích hợp tiếp khách"),

    ("🍽️ Giá rẻ", 
     "Quán ăn ngon bổ rẻ cho sinh viên ở Đà Nẵng"),

    ("🍽️ Gia đình", 
     "Nhà hàng phù hợp cho gia đình có trẻ em"),

    ("🍽️ Mở khuya", 
     "Quán ăn nào mở khuya ở Đà Nẵng?"),

 
    # ==================================================
    # ĐỊA ĐIỂM DU LỊCH
    # ==================================================

    ("📍 Địa điểm", 
     "Địa điểm du lịch nổi tiếng nào ở bán đảo Sơn Trà?"),

    ("📍 Check-in", 
     "Địa điểm check-in đẹp ở Đà Nẵng"),

    ("📍 Biển", 
     "Bãi biển nào đẹp nhất ở Đà Nẵng?"),

    ("📍 Ban đêm", 
     "Buổi tối ở Đà Nẵng nên đi đâu chơi?"),

    ("📍 Thiên nhiên", 
     "Địa điểm thiên nhiên đẹp và yên bình"),

    ("📍 Văn hóa", 
     "Các địa điểm mang nét văn hóa và lịch sử ở Đà Nẵng"),

    ("📍 Gia đình", 
     "Địa điểm vui chơi phù hợp cho gia đình có trẻ em"),

    ("📍 Miễn phí", 
     "Địa điểm tham quan miễn phí ở Đà Nẵng"),


    ("📍 Cuối tuần", 
     "Cuối tuần nên đi đâu ở Đà Nẵng?"),

    ("📍 Chụp ảnh", 
     "Địa điểm chụp ảnh đẹp lúc hoàng hôn"),

  

    # ==================================================
    # Ý ĐỊNH PHỨC TẠP
    # ==================================================


    ("🧠 Kết hợp", 
     "Gợi ý khách sạn gần biển và có nhà hàng hải sản ngon xung quanh"),

    ("🧠 Theo nhu cầu", 
     "Tôi thích nơi yên tĩnh, ít đông người, nên ở đâu?"),

    

   
]

for category, query in test_queries:
    print(f"\n{'='*80}")
    print(f"[{category}]")
    print(f"Câu hỏi: {query}")
    print("-" * 80)
    
    try:
        # Sử dụng filter parser nếu đã thêm
        result = pipeline.answer(query, top_k=4, temperature=0.2)
        
        print(f"\n📊 Intent phát hiện: {result['intent']}")
        print(f"\n🤖 Trả lời:\n{result['answer']}")
        
        print(f"\n📋 Kết quả tìm thấy ({result['total_results']} items):")
        for i, r in enumerate(result['results'][:5], 1):
            print(f"\n  {i}. {r['name']}")
            print(f"     - Loại: {r['type']}")
            if r.get('district'):
                print(f"     - Quận: {r['district']}")
            if r.get('rating'):
                print(f"     - Đánh giá: {r['rating']}")
            if r.get('price') and r['price'] != "Không có thông tin giá":
                print(f"     - Giá: {r['price']}")
            if r.get('address'):
                print(f"     - Địa chỉ: {r['address'][:100]}...")
                
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
    
    print("-" * 80)

print("\n" + "="*80)
print("DEMO HOÀN TẤT")
print("="*80)

RAG SYSTEM DEMO - ĐÀ NẴNG TRAVEL ASSISTANT
🔄 Khởi tạo RAG Pipeline...
✅ Pipeline initialized successfully!

[🏨 Khách sạn]
Câu hỏi: Gợi ý khách sạn ở quận Sơn Trà có đánh giá tốt trên 8.0
--------------------------------------------------------------------------------

📊 Intent phát hiện: hotel_search

🤖 Trả lời:
Dựa trên yêu cầu của bạn, tôi gợi ý cho bạn một số khách sạn ở quận Sơn Trà với đánh giá tốt trên 8.0:

1. **Saha Casa Boutique Hotel Da Nang**
   - Đánh giá: 9.6/10 (20 đánh giá)
   - Địa chỉ: 10 Morrison Son Trà, Đà Nẵng, Việt Nam, 550000
   
   Saha Casa Boutique Hotel Da Nang có đánh giá rất cao, đạt 9.6/10 từ 20 đánh giá. Khách sạn này nằm ở quận Sơn Trà, Đà Nẵng và được đánh giá cao về chất lượng dịch vụ và không gian nghỉ ngơi.

2. **Hotel Le Bouton**
   - Đánh giá: 9.1/10 (86 đánh giá)
   - Địa chỉ: 12 Trần Đình Đàn, Sơn Trà, Đà Nẵng, Việt Nam, 550000
   
   Hotel Le Bouton cũng có đánh giá cao, đạt 9.1/10 từ 86 đánh giá. Khách sạn này nằm ở quận Sơn Trà, Đà Nẵng và đượ

In [8]:
# ============================================
# CELL 8: Advanced Filtered Search Demo (ĐÃ CHỈNH SỬA)
# ============================================

print("\n" + "="*80)
print("ADVANCED FILTERED SEARCH DEMO")
print("="*80)

# Đảm bảo pipeline tồn tại
if 'pipeline' not in globals() or pipeline is None:
    print("🔄 Khởi tạo RAG Pipeline...")
    pipeline = RAGPipeline()
    print("✅ Pipeline initialized successfully!")

# Demo with filters
filtered_queries = [
    ("Tìm khách sạn cao cấp Sơn Trà", 
     "Khách sạn ở Sơn Trà", 
     {"district": "son tra", "min_rating": 8.0}),
    
    ("Tìm nhà hàng Hải Châu giá rẻ", 
     "Nhà hàng ở Hải Châu", 
     {"district": "hai chau", "max_price": 500000}),
    
    ("Khách sạn dưới 800k ở Sơn Trà", 
     "Khách sạn Sơn Trà giá rẻ", 
     {"district": "son tra", "max_price": 800000}),
    
    ("Nhà hàng hải sản ngon dưới 1 triệu", 
     "Nhà hàng hải sản", 
     {"max_price": 1000000}),
]

for description, query, filters in filtered_queries:
    print(f"\n📌 {description}")
    print(f"Query: {query}")
    print(f"Filters: {filters}")
    print("-" * 70)
    
    try:
        result = pipeline.answer(
            query=query, 
            top_k=4, 
            filters=filters, 
            temperature=0.2
        )
        
        print(f"📊 Intent: {result['intent']}")
        print(f"🔍 Tìm thấy {result['total_results']} kết quả")
        
        print(f"\n🤖 Trả lời:\n{result['answer']}")
        
        print(f"\n📋 Top kết quả:")
        for i, r in enumerate(result['results'][:5], 1):
            print(f"\n  {i}. {r['name']}")
            print(f"     • Loại     : {r['type']}")
            if r.get('district'):
                print(f"     • Quận     : {r['district']}")
            if r.get('rating'):
                print(f"     • Đánh giá : {r['rating']}")
            if r.get('price') and r['price'] != "Không có thông tin giá":
                print(f"     • Giá      : {r['price']}")
            print(f"     • Score    : {r['score']:.3f}")
            
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()
    
    print("-" * 70)

print("\n" + "="*80)
print("ADVANCED FILTERED DEMO HOÀN TẤT")
print("="*80)


ADVANCED FILTERED SEARCH DEMO

📌 Tìm khách sạn cao cấp Sơn Trà
Query: Khách sạn ở Sơn Trà
Filters: {'district': 'son tra', 'min_rating': 8.0}
----------------------------------------------------------------------
📊 Intent: hotel_search
🔍 Tìm thấy 4 kết quả

🤖 Trả lời:
Dựa trên thông tin cung cấp, đây là một số khách sạn tốt ở Sơn Trà, Đà Nẵng:

1. **Hotel Le Bouton**
   - Đánh giá: 9.1/10 (86 đánh giá)
   - Giá: 1,042,801 - 1,215,035 VND
   - Địa chỉ: 12 Trần Đình Đàn, Sơn Trà, Đà Nẵng

   Khách sạn này nằm trong khu vực Son Tra, có vị trí thuận lợi và được đánh giá cao. 

2. **Sala Danang Beach Hotel**
   - Đánh giá: 9.4/10 (2,581 đánh giá)
   - Giá: 500,000 VND
   - Địa chỉ: 36-38 Lam Hoanh, Phước Mỹ, Sơn Trà, Đà Nẵng

   Đây là một trong những khách sạn tốt nhất ở Son Tra với đánh giá cao và vị trí đẹp.

3. **Son Tra Beach Resort & Spa Danang**
   - Đánh giá: 8.5/10 (614 đánh giá)
   - Giá: 900,000 VND
   - Địa chỉ: Nam Son Tra Beach, Đà Nẵng

   Khách sạn này cũng nằm ở Son Tra và